# Lab01 — Demo phân loại email spam
Chạy các ô theo thứ tự. Notebook dùng ba class tự cài đặt và các mô hình đã huấn luyện trên TF-IDF.
Nếu chưa có mô hình, chạy `python -m Lab01.src.train` từ thư mục gốc rồi quay lại notebook.
Nhãn 0 là ham, nhãn 1 là spam.

In [ ]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Lab01" / "algorithms").is_dir()), None)
if root is None:
    raise RuntimeError("Open the notebook inside the repository.")
sys.path.insert(0, str(root))

import joblib
import pandas as pd
from Lab01.algorithms import LogisticRegression, LinearSVM, MultinomialNaiveBayes
from Lab01.src.preprocessing import clean_text

lab = root / "Lab01"
artifacts = lab / "models"
vectorizer = joblib.load(artifacts / "tfidf_vectorizer.joblib")

## Logistic Regression
Sigmoid chuyển điểm tuyến tính thành xác suất spam. Ngưỡng dự đoán là 0,5.

In [ ]:
logistic = joblib.load(artifacts / "logistic_regression.joblib")
assert isinstance(logistic, LogisticRegression)

## Linear SVM
Hinge loss học ranh giới tuyến tính. Điểm từ ngưỡng đã lưu trở lên được gán spam.

In [ ]:
svm = joblib.load(artifacts / "svm.joblib")
assert isinstance(svm, LinearSVM)

## Multinomial Naive Bayes
Cộng điểm log của các đặc trưng và xác suất tiên nghiệm rồi chọn lớp có điểm cao hơn.

In [ ]:
bayes = joblib.load(artifacts / "naive_bayes.joblib")
assert isinstance(bayes, MultinomialNaiveBayes)

## Demo trên email mới
Thay nội dung trong danh sách để thử thêm. Cả ba mô hình dùng chung ma trận TF-IDF dạng thưa.

In [ ]:
emails = [
    "Please review the attached agenda for tomorrow team meeting",
    "Congratulations! Claim your free cash prize now",
]
X = vectorizer.transform([clean_text(email) for email in emails])
labels = {0: "ham", 1: "spam"}
demo = pd.DataFrame({
    "Email": emails,
    "Logistic Regression": [labels[p] for p in logistic.predict(X)],
    "Spam probability (LR)": logistic.predict_proba(X),
    "Linear SVM": [labels[p] for p in svm.predict(X)],
    "Decision score (SVM)": svm.decision_function(X),
    "SVM threshold": svm.threshold,
    "Naive Bayes": [labels[p] for p in bayes.predict(X)],
})
demo

## So sánh trên tập đối chiếu đã xem
Macro F1 lấy trung bình F1 của ham và spam. Các email demo ở trên minh họa cách dự đoán.

In [ ]:
pd.read_csv(lab / "result/comparison/metrics.csv")